# Product Recommendation System - Collaborative Filtering
## Building a Smart Recommendation Engine for Amazon Products

This notebook demonstrates how to build a collaborative filtering-based product recommendation system using user-item interaction data.

## 1. Import Required Libraries

In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial.distance import euclidean
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

## 2. Load and Explore the Data

In [ ]:
# Load the Amazon products data
df = pd.read_csv('Data/amazon.csv')

print("Dataset Shape:", df.shape)
print("\nFirst few rows:")
print(df.head())

print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

In [ ]:
# Exploratory Data Analysis
print("=" * 60)
print("EXPLORATORY DATA ANALYSIS")
print("=" * 60)

# User and Product Statistics
print(f"\n📊 Dataset Statistics:")
print(f"  • Total Reviews: {len(df)}")
print(f"  • Unique Users: {df['user_id'].nunique()}")
print(f"  • Unique Products: {df['product_id'].nunique()}")
print(f"  • Unique Categories: {df['category'].nunique()}")

# Rating Distribution
print(f"\n⭐ Rating Distribution:")
print(df['rating'].value_counts().sort_index())

# Visualize rating distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Rating distribution histogram
df['rating'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Distribution of Ratings', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Frequency')

# Category distribution
df['category'].value_counts().head(10).plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Top 10 Product Categories', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.show()

# Data sparsity
sparsity = 1 - (len(df) / (df['user_id'].nunique() * df['product_id'].nunique()))
print(f"\n🔍 Data Sparsity: {sparsity*100:.2f}%")
print(f"   (Lower sparsity = more interaction data = better recommendations)")

## 3. Prepare Data for Collaborative Filtering

In [ ]:
# Data Cleaning and Preprocessing
df_clean = df.copy()

# Remove duplicates
initial_rows = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=['product_id', 'user_id'], keep='first')
print(f"✓ Removed {initial_rows - len(df_clean)} duplicate records")

# Ensure rating is numeric
df_clean['rating'] = pd.to_numeric(df_clean['rating'], errors='coerce')
df_clean = df_clean.dropna(subset=['rating'])

# Filter invalid ratings
df_clean = df_clean[(df_clean['rating'] >= 1) & (df_clean['rating'] <= 5)]
print(f"✓ Kept {len(df_clean)} valid ratings")

# Filter out sparse data (users/products with too few interactions)
min_user_ratings = 2
min_product_ratings = 2

user_counts = df_clean['user_id'].value_counts()
valid_users = user_counts[user_counts >= min_user_ratings].index
df_clean = df_clean[df_clean['user_id'].isin(valid_users)]
print(f"✓ Kept users with at least {min_user_ratings} ratings: {len(df_clean)} records")

product_counts = df_clean['product_id'].value_counts()
valid_products = product_counts[product_counts >= min_product_ratings].index
df_clean = df_clean[df_clean['product_id'].isin(valid_products)]
print(f"✓ Kept products with at least {min_product_ratings} ratings: {len(df_clean)} records")

print(f"\n📊 Final Dataset:")
print(f"  • Records: {len(df_clean)}")
print(f"  • Users: {df_clean['user_id'].nunique()}")
print(f"  • Products: {df_clean['product_id'].nunique()}")
print(f"  • Rating Mean: {df_clean['rating'].mean():.2f}")
print(f"  • Rating Std Dev: {df_clean['rating'].std():.2f}")

## 4. Build User-Item Interaction Matrix

In [ ]:
# Create User-Item Rating Matrix
# Rows = Users, Columns = Products, Values = Ratings

user_item_matrix = df_clean.pivot_table(
    index='user_id',
    columns='product_id',
    values='rating',
    aggfunc='mean'
)

# Fill NaN with 0 (no rating)
user_item_matrix = user_item_matrix.fillna(0)

print("User-Item Interaction Matrix:")
print(f"  • Shape: {user_item_matrix.shape}")
print(f"  • Users (rows): {user_item_matrix.shape[0]}")
print(f"  • Products (columns): {user_item_matrix.shape[1]}")

# Calculate sparsity
sparsity = 1 - (len(df_clean) / (user_item_matrix.shape[0] * user_item_matrix.shape[1]))
print(f"  • Sparsity: {sparsity*100:.2f}%")
print(f"  • Density: {(1-sparsity)*100:.2f}%")

# Show sample of the matrix
print("\nSample of User-Item Matrix (first 5 users, first 5 products):")
print(user_item_matrix.iloc[:5, :5])

# Visualize sparsity
plt.figure(figsize=(10, 6))
plt.spy(user_item_matrix > 0, markersize=0.5)
plt.title('User-Item Rating Matrix Sparsity Pattern', fontsize=12, fontweight='bold')
plt.xlabel('Products')
plt.ylabel('Users')
plt.tight_layout()
plt.show()

## 5. Calculate Similarity Metrics

In [ ]:
# Compute User-to-User Similarity (Cosine Similarity)
print("Computing User-to-User Similarity using Cosine Similarity...")
user_similarity = cosine_similarity(user_item_matrix)
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)
print(f"✓ User Similarity Matrix Shape: {user_similarity_df.shape}")

# Compute Item-to-Item Similarity (Cosine Similarity)
print("\nComputing Item-to-Item Similarity using Cosine Similarity...")
item_similarity = cosine_similarity(user_item_matrix.T)
item_similarity_df = pd.DataFrame(
    item_similarity,
    index=user_item_matrix.columns,
    columns=user_item_matrix.columns
)
print(f"✓ Item Similarity Matrix Shape: {item_similarity_df.shape}")

# Display sample similarity scores
print("\nSample User-to-User Similarity (first 5 users):")
print(user_similarity_df.iloc[:5, :5].round(3))

print("\nSample Item-to-Item Similarity (first 5 products):")
print(item_similarity_df.iloc[:5, :5].round(3))

# Visualize similarity distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# User similarity distribution
user_sim_values = user_similarity[np.triu_indices_from(user_similarity, k=1)]
axes[0].hist(user_sim_values, bins=30, color='steelblue', edgecolor='black')
axes[0].set_title('Distribution of User-to-User Similarity', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Similarity Score')
axes[0].set_ylabel('Frequency')

# Item similarity distribution
item_sim_values = item_similarity[np.triu_indices_from(item_similarity, k=1)]
axes[1].hist(item_sim_values, bins=30, color='coral', edgecolor='black')
axes[1].set_title('Distribution of Item-to-Item Similarity', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Similarity Score')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## 6. Implement User-Based Collaborative Filtering

**How it works:**
1. Find users most similar to the target user
2. Look at products rated highly by these similar users
3. Recommend products the target user hasn't rated yet

In [ ]:
def recommend_user_based(user_id, n_recommendations=5, n_similar_users=5):
    """
    Recommend products using User-Based Collaborative Filtering
    
    Args:
        user_id: The user to make recommendations for
        n_recommendations: Number of products to recommend
        n_similar_users: Number of similar users to consider
    
    Returns:
        DataFrame with recommended products and predicted ratings
    """
    if user_id not in user_similarity_df.index:
        return pd.DataFrame()
    
    # Get similar users
    similar_users = user_similarity_df[user_id].sort_values(ascending=False)[1:n_similar_users+1]
    
    if len(similar_users) == 0:
        return pd.DataFrame()
    
    # Get ratings from similar users
    similar_users_ratings = user_item_matrix.loc[similar_users.index]
    
    # Weighted average of ratings from similar users
    weighted_ratings = similar_users_ratings.T.dot(similar_users.values) / similar_users.sum()
    
    # Products already rated by the user
    user_rated = user_item_matrix.loc[user_id]
    user_rated_products = user_rated[user_rated > 0].index
    
    # Exclude products already rated
    recommendations = weighted_ratings.drop(user_rated_products, errors='ignore')
    recommendations = recommendations.sort_values(ascending=False)[:n_recommendations]
    
    return pd.DataFrame({
        'product_id': recommendations.index,
        'predicted_rating': recommendations.values
    })

# Test with a sample user
sample_user = df_clean['user_id'].iloc[0]
print(f"🎯 Making User-Based Recommendations for User: {sample_user}\n")

user_based_recs = recommend_user_based(sample_user, n_recommendations=5, n_similar_users=5)

if not user_based_recs.empty:
    print(f"Top 5 Recommended Products (Predicted Ratings):")
    for idx, row in user_based_recs.iterrows():
        product_info = df_clean[df_clean['product_id'] == row['product_id']].iloc[0]
        print(f"\n  {idx+1}. {product_info['product_name'][:60]}")
        print(f"     Product ID: {row['product_id']}")
        print(f"     Predicted Rating: {row['predicted_rating']:.2f}/5.0")
        print(f"     Category: {product_info['category']}")
        print(f"     Price: {product_info['discounted_price']}")
else:
    print("No recommendations available for this user.")

## 7. Implement Item-Based Collaborative Filtering

**How it works:**
1. Find products similar to those the user has rated highly
2. Score unrated products based on similarity
3. Recommend top-scored products

In [ ]:
def recommend_item_based(user_id, n_recommendations=5):
    """
    Recommend products using Item-Based Collaborative Filtering
    
    Args:
        user_id: The user to make recommendations for
        n_recommendations: Number of products to recommend
    
    Returns:
        DataFrame with recommended products and predicted ratings
    """
    if user_id not in user_item_matrix.index:
        return pd.DataFrame()
    
    # Get user's ratings
    user_ratings = user_item_matrix.loc[user_id]
    
    # Get products the user has rated
    rated_products = user_ratings[user_ratings > 0]
    
    if len(rated_products) == 0:
        return pd.DataFrame()
    
    # Calculate scores for unrated products
    recommendation_scores = {}
    
    for product_id in user_item_matrix.columns:
        if product_id not in rated_products.index:
            # Get similarity scores for this product with all rated products
            similarities = item_similarity_df.loc[product_id, rated_products.index].values
            ratings = rated_products.values
            
            # Weighted average
            if similarities.sum() > 0:
                score = (similarities * ratings).sum() / similarities.sum()
                recommendation_scores[product_id] = score
    
    # Sort and return top recommendations
    recommendations = sorted(recommendation_scores.items(), 
                            key=lambda x: x[1], reverse=True)[:n_recommendations]
    
    return pd.DataFrame({
        'product_id': [r[0] for r in recommendations],
        'predicted_rating': [r[1] for r in recommendations]
    })

# Test with the same sample user
print(f"🎯 Making Item-Based Recommendations for User: {sample_user}\n")

item_based_recs = recommend_item_based(sample_user, n_recommendations=5)

if not item_based_recs.empty:
    print(f"Top 5 Recommended Products (Predicted Ratings):")
    for idx, row in item_based_recs.iterrows():
        product_info = df_clean[df_clean['product_id'] == row['product_id']].iloc[0]
        print(f"\n  {idx+1}. {product_info['product_name'][:60]}")
        print(f"     Product ID: {row['product_id']}")
        print(f"     Predicted Rating: {row['predicted_rating']:.2f}/5.0")
        print(f"     Category: {product_info['category']}")
        print(f"     Price: {product_info['discounted_price']}")
else:
    print("No recommendations available for this user.")

## 8. Generate Comprehensive Recommendations

**Hybrid Approach:** Combine both user-based and item-based recommendations

In [ ]:
def hybrid_recommend(user_id, n_recommendations=5, user_weight=0.5, item_weight=0.5):
    """
    Hybrid recommendation combining user-based and item-based approaches
    
    Args:
        user_id: The user to make recommendations for
        n_recommendations: Number of products to recommend
        user_weight: Weight for user-based recommendations
        item_weight: Weight for item-based recommendations
    
    Returns:
        DataFrame with hybrid recommendations
    """
    user_recs = recommend_user_based(user_id, n_recommendations * 2)
    item_recs = recommend_item_based(user_id, n_recommendations * 2)
    
    # Combine scores
    all_scores = {}
    
    if not user_recs.empty:
        user_max = user_recs['predicted_rating'].max()
        for _, row in user_recs.iterrows():
            all_scores[row['product_id']] = user_weight * (row['predicted_rating'] / user_max if user_max > 0 else 0)
    
    if not item_recs.empty:
        item_max = item_recs['predicted_rating'].max()
        for _, row in item_recs.iterrows():
            if row['product_id'] in all_scores:
                all_scores[row['product_id']] += item_weight * (row['predicted_rating'] / item_max if item_max > 0 else 0)
            else:
                all_scores[row['product_id']] = item_weight * (row['predicted_rating'] / item_max if item_max > 0 else 0)
    
    # Sort and return
    recommendations = sorted(all_scores.items(), 
                            key=lambda x: x[1], reverse=True)[:n_recommendations]
    
    return pd.DataFrame({
        'product_id': [r[0] for r in recommendations],
        'hybrid_score': [r[1] for r in recommendations]
    })

# Generate recommendations for multiple users
print("=" * 70)
print("HYBRID RECOMMENDATIONS FOR MULTIPLE USERS")
print("=" * 70)

sample_users = df_clean['user_id'].unique()[:3]

for user_id in sample_users:
    print(f"\n\n👤 USER: {user_id}")
    print("-" * 70)
    
    hybrid_recs = hybrid_recommend(user_id, n_recommendations=3)
    
    if not hybrid_recs.empty:
        for idx, row in hybrid_recs.iterrows():
            product_info = df_clean[df_clean['product_id'] == row['product_id']].iloc[0]
            print(f"\n  Rank {idx+1}: {product_info['product_name'][:55]}")
            print(f"    Hybrid Score: {row['hybrid_score']:.3f}")
            print(f"    Category: {product_info['category'][:50]}")
    else:
        print("  No recommendations available")

## 9. Evaluate Recommendation Quality

**Metrics:**
- **Precision@K**: What fraction of top-K recommendations are relevant
- **Recall@K**: What fraction of all relevant items are in top-K recommendations
- **Mean Reciprocal Rank (MRR)**: How highly ranked is the first relevant item

In [ ]:
# Evaluation Functions
def calculate_precision_at_k(recommendations, relevant_items, k=5):
    """Calculate Precision@K"""
    if len(recommendations) == 0:
        return 0.0
    rec_at_k = recommendations[:k]
    relevant_in_rec = len([r for r in rec_at_k if r in relevant_items])
    return relevant_in_rec / k if k > 0 else 0.0

def calculate_recall_at_k(recommendations, relevant_items, k=5):
    """Calculate Recall@K"""
    if len(relevant_items) == 0:
        return 0.0
    rec_at_k = recommendations[:k]
    relevant_in_rec = len([r for r in rec_at_k if r in relevant_items])
    return relevant_in_rec / len(relevant_items)

# Simple evaluation: Use high-rated products as relevant items
high_rating_threshold = 4.0
k = 5

precision_scores = []
recall_scores = []

print(f"\n📊 Evaluating Recommendations (K={k}, High Rating Threshold={high_rating_threshold})")
print("=" * 70)

for user_id in df_clean['user_id'].unique()[:20]:  # Test on first 20 users
    # Get user's high-rated products
    user_data = df_clean[df_clean['user_id'] == user_id]
    high_rated = user_data[user_data['rating'] >= high_rating_threshold]['product_id'].tolist()
    
    if len(high_rated) >= 2:
        # Get recommendations
        recs = recommend_user_based(user_id, n_recommendations=k)
        
        if not recs.empty:
            rec_products = recs['product_id'].tolist()
            
            # Calculate metrics
            precision = calculate_precision_at_k(rec_products, high_rated, k=k)
            recall = calculate_recall_at_k(rec_products, high_rated, k=k)
            
            precision_scores.append(precision)
            recall_scores.append(recall)

# Calculate average metrics
if precision_scores:
    avg_precision = np.mean(precision_scores)
    avg_recall = np.mean(recall_scores)
    
    print(f"\n✅ Evaluation Results:")
    print(f"  • Average Precision@{k}: {avg_precision:.3f}")
    print(f"  • Average Recall@{k}: {avg_recall:.3f}")
    print(f"  • F1-Score: {2 * (avg_precision * avg_recall) / (avg_precision + avg_recall) if (avg_precision + avg_recall) > 0 else 0:.3f}")
    
    # Visualize evaluation metrics
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    axes[0].hist(precision_scores, bins=10, color='steelblue', edgecolor='black')
    axes[0].axvline(avg_precision, color='red', linestyle='--', linewidth=2, label=f'Mean: {avg_precision:.3f}')
    axes[0].set_title(f'Precision@{k} Distribution', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Precision Score')
    axes[0].set_ylabel('Frequency')
    axes[0].legend()
    
    axes[1].hist(recall_scores, bins=10, color='coral', edgecolor='black')
    axes[1].axvline(avg_recall, color='red', linestyle='--', linewidth=2, label=f'Mean: {avg_recall:.3f}')
    axes[1].set_title(f'Recall@{k} Distribution', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Recall Score')
    axes[1].set_ylabel('Frequency')
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()
else:
    print("Unable to calculate metrics - insufficient data")

## Summary & Key Insights

### 🎯 What We've Built

A **Collaborative Filtering Recommendation System** with three approaches:

1. **User-Based CF**: Recommends products liked by similar users
   - Pros: Captures user preferences well
   - Cons: Sensitive to sparsity, new user problem
   
2. **Item-Based CF**: Recommends products similar to user's liked items  
   - Pros: Works well for products with clear patterns
   - Cons: Cold-start for new items

3. **Hybrid Approach**: Combines both methods for better coverage
   - Pros: More robust, fewer cold-start issues
   - Cons: Higher computational cost

### 📈 Performance Considerations

- **Sparsity**: Most user-item interactions are missing (sparse data)
- **Cold Start**: New users/items with no ratings are hard to recommend for
- **Scalability**: Computing full similarity matrices doesn't scale well for massive datasets

### 🚀 Next Steps for Production

1. **Use Advanced Techniques**:
   - Matrix Factorization (SVD, NMF)
   - Deep Learning (Neural Collaborative Filtering)
   - Graph-based methods
   
2. **Address Cold Start**:
   - Content-based features
   - Meta-data (categories, brands)
   - New user onboarding
   
3. **Optimize at Scale**:
   - Approximate nearest neighbor search
   - Distributed computing (Spark)
   - Real-time recommendation APIs

4. **Monitor & Improve**:
   - A/B testing
   - User feedback loops
   - Continuous model updates